Most of the initial steps are similar to customer processing


In [0]:
spark

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
dbutils.widgets.text("catalog", "consumer_goods", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")
base_path = f'/Volumes/consumer_goods/files/input_data_files/child_company/{data_source}/*.csv'

print(catalog)
print(data_source)
print(base_path)


consumer_goods
products
/Volumes/consumer_goods/files/input_data_files/child_company/products/*.csv


In [0]:
df = (spark.read.format("csv")
        .option("header", "True")
        .option("inferSchema", "True")
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*","_metadata.file_name")
)
df.show(10)

+--------------------+----------+-----------------+--------------------+------------+
|        product_name|product_id|         category|      read_timestamp|   file_name|
+--------------------+----------+-----------------+--------------------+------------+
|SportsBar Energy ...|  25891101|      energy bars|2026-03-03 09:03:...|products.csv|
|SportsBar Energy ...|  25891102|      energy bars|2026-03-03 09:03:...|products.csv|
|SportsBar Energy ...|  25891103|      energy bars|2026-03-03 09:03:...|products.csv|
|SportsBar Protien...|  25891201|     protien bars|2026-03-03 09:03:...|products.csv|
|SportsBar Protien...|  25891202|     protien bars|2026-03-03 09:03:...|products.csv|
|SportsBar Protien...|  25891203|     protien bars|2026-03-03 09:03:...|products.csv|
|SportsBar Granola...|  25891301|granola & cereals|2026-03-03 09:03:...|products.csv|
|SportsBar Granola...|  25891302|granola & cereals|2026-03-03 09:03:...|products.csv|
|SportsBar Granola...|  25891303|granola & cereals|202

In [0]:
df.write.format("delta").option("delta.enableChangeDataFeed", "true").mode("overwrite").saveAsTable(f"{catalog}.bronze.{data_source}")

dropping duplicates

In [0]:
df_no_duplicates = df.dropDuplicates(["product_id"])
print("Before :",df.count())
print("After :",df_no_duplicates.count())

Before : 20
After : 18


case fixing

In [0]:
df_no_duplicates = df_no_duplicates.withColumn("category",
    F.when(F.col("category").isNull(), None).otherwise(F.initcap("category"))
)
display(df_no_duplicates)

product_name,product_id,category,read_timestamp,file_name
SportsBar Energy Bar Choco Fudge (60g),25891101,Energy Bars,2026-03-03T09:03:07.434Z,products.csv
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-03-03T09:03:07.434Z,products.csv
SportsBar Energy Bar Choco Fudge (25g),25891103,Energy Bars,2026-03-03T09:03:07.434Z,products.csv
SportsBar Protien Bar Peanut Crunch (45g),25891201,Protien Bars,2026-03-03T09:03:07.434Z,products.csv
SportsBar Protien Bar Peanut Crunch (55g),25891202,Protien Bars,2026-03-03T09:03:07.434Z,products.csv
SportsBar Protien Bar Peanut Crunch (65g),25891203,Protien Bars,2026-03-03T09:03:07.434Z,products.csv
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-03-03T09:03:07.434Z,products.csv
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-03-03T09:03:07.434Z,products.csv
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-03-03T09:03:07.434Z,products.csv
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,Recovery Dairy,2026-03-03T09:03:07.434Z,products.csv


Protein spelling fix
(?i): This is a regex flag for "case-insensitive." It ensures that "Protien", "PROTIEN", and "protien" are all matched.

In [0]:
df_no_duplicates = (
    df_no_duplicates.withColumn("product_name",
        F.regexp_replace(F.col("product_name"), "(?i)Protien", "Protein")
    )
    .withColumn("category",
        F.regexp_replace(F.col("category"), "(?i)Protien", "Protein")
    )
)

Division names in Main table has difference wrt child company

In [0]:
df_no_duplicates = (
    df_no_duplicates
    .withColumn(
        "division",
        F.when(F.col("category") == "Energy Bars",        "Nutrition Bars")
         .when(F.col("category") == "Protein Bars",       "Nutrition Bars")
         .when(F.col("category") == "Granola & Cereals",  "Breakfast Foods")
         .when(F.col("category") == "Recovery Dairy",     "Dairy & Recovery")
         .when(F.col("category") == "Healthy Snacks",     "Healthy Snacks")
         .when(F.col("category") == "Electrolyte Mix",    "Hydration & Electrolytes")
         .otherwise("Other")
    )
)
display(df_no_duplicates)

product_name,product_id,category,read_timestamp,file_name,division
SportsBar Energy Bar Choco Fudge (60g),25891101,Energy Bars,2026-03-03T09:03:08.703Z,products.csv,Nutrition Bars
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-03-03T09:03:08.703Z,products.csv,Nutrition Bars
SportsBar Energy Bar Choco Fudge (25g),25891103,Energy Bars,2026-03-03T09:03:08.703Z,products.csv,Nutrition Bars
SportsBar Protein Bar Peanut Crunch (45g),25891201,Protein Bars,2026-03-03T09:03:08.703Z,products.csv,Nutrition Bars
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protein Bars,2026-03-03T09:03:08.703Z,products.csv,Nutrition Bars
SportsBar Protein Bar Peanut Crunch (65g),25891203,Protein Bars,2026-03-03T09:03:08.703Z,products.csv,Nutrition Bars
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-03-03T09:03:08.703Z,products.csv,Breakfast Foods
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-03-03T09:03:08.703Z,products.csv,Breakfast Foods
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-03-03T09:03:08.703Z,products.csv,Breakfast Foods
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,Recovery Dairy,2026-03-03T09:03:08.703Z,products.csv,Dairy & Recovery


Creating a variant column out of name.
- Matches and find data inside ()

In [0]:
df_no_duplicates = df_no_duplicates.withColumn("variant",
    F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
)
display(df_no_duplicates)

product_name,product_id,category,read_timestamp,file_name,division,variant
SportsBar Energy Bar Choco Fudge (60g),25891101,Energy Bars,2026-03-03T09:03:09.749Z,products.csv,Nutrition Bars,60g
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-03-03T09:03:09.749Z,products.csv,Nutrition Bars,40g
SportsBar Energy Bar Choco Fudge (25g),25891103,Energy Bars,2026-03-03T09:03:09.749Z,products.csv,Nutrition Bars,25g
SportsBar Protein Bar Peanut Crunch (45g),25891201,Protein Bars,2026-03-03T09:03:09.749Z,products.csv,Nutrition Bars,45g
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protein Bars,2026-03-03T09:03:09.749Z,products.csv,Nutrition Bars,55g
SportsBar Protein Bar Peanut Crunch (65g),25891203,Protein Bars,2026-03-03T09:03:09.749Z,products.csv,Nutrition Bars,65g
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-03-03T09:03:09.749Z,products.csv,Breakfast Foods,400g
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-03-03T09:03:09.749Z,products.csv,Breakfast Foods,300g
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-03-03T09:03:09.749Z,products.csv,Breakfast Foods,200g
SportsBar Greek Yogurt Pro Vanilla (200g),25891401,Recovery Dairy,2026-03-03T09:03:09.749Z,products.csv,Dairy & Recovery,200g


Creating product_code

In [0]:
df_no_duplicates = (
    df_no_duplicates
    # 1. Generate deterministic product_code from product_name
    .withColumn(
        "product_code",
        F.sha2(F.col("product_name").cast("string"), 256)
    )
    # 2. Clean product_id: keep only numeric IDs, else set to 999999
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit(999999).cast("string"))
    )
    # 3. Rename product_name → product
    .withColumnRenamed("product_name", "product")
)
display(df_no_duplicates)

product,product_id,category,read_timestamp,file_name,division,variant,product_code
SportsBar Oats Cookie Bites ChocoChip (180g),25891503,Healthy Snacks,2026-03-03T09:03:10.744Z,products.csv,Healthy Snacks,180g,062f5574bbdf4386b2c7c6075483b417b4a00b172fcba919dbba7dae1b774379
SportsBar Energy Bar Choco Fudge (40g),25891102,Energy Bars,2026-03-03T09:03:10.744Z,products.csv,Nutrition Bars,40g,e92c739a8d78cd6cbe954648c2f9dd75ed61fcfd99b03e10dca65c3082d0728e
SportsBar Granola Crunch Honey Almond (300g),25891302,Granola & Cereals,2026-03-03T09:03:10.744Z,products.csv,Breakfast Foods,300g,d9ebd1ca64d23951a6310af93b1c5ac27d831ac842e89aea59a9e8b38621faa5
SportsBar Greek Yogurt Pro Vanilla (80g),25891403,Recovery Dairy,2026-03-03T09:03:10.744Z,products.csv,Dairy & Recovery,80g,77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9
SportsBar Granola Crunch Honey Almond (400g),25891301,Granola & Cereals,2026-03-03T09:03:10.744Z,products.csv,Breakfast Foods,400g,3cab59f05924285270313afcfe40a08983bb03dd88f432e34fc6336914c14345
SportsBar Oats Cookie Bites ChocoChip (350g),999999,Healthy Snacks,2026-03-03T09:03:10.744Z,products.csv,Healthy Snacks,350g,5931334e4cbe6b3792c209e8394e87aa21b83816b47d99375e4ff25e651ce63a
SportsBar Protein Bar Peanut Crunch (55g),25891202,Protein Bars,2026-03-03T09:03:10.744Z,products.csv,Nutrition Bars,55g,0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28
SportsBar Electrolyte Mix Lemon-Lime (30 Sachets),25891601,Electrolyte Mix,2026-03-03T09:03:10.744Z,products.csv,Hydration & Electrolytes,30 Sachets,716fa4e54b7894c910180276e0535d49afb25cdcfac09533fb74ae00689e5742
SportsBar Electrolyte Mix Lemon-Lime (15 Sachets),25891602,Electrolyte Mix,2026-03-03T09:03:10.744Z,products.csv,Hydration & Electrolytes,15 Sachets,778c2a7aa27bfdb211fd5ece048de80d00fbf3d6924bd908d91054796ba16ab6
SportsBar Granola Crunch Honey Almond (200g),25891303,Granola & Cereals,2026-03-03T09:03:10.744Z,products.csv,Breakfast Foods,200g,c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f


In [0]:
df_silver = df_no_duplicates.select("product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "file_name",)
df_silver.show(10)

+--------------------+--------------------+-----------------+--------------------+----------+----------+--------------------+------------+
|        product_code|            division|         category|             product|   variant|product_id|      read_timestamp|   file_name|
+--------------------+--------------------+-----------------+--------------------+----------+----------+--------------------+------------+
|062f5574bbdf4386b...|      Healthy Snacks|   Healthy Snacks|SportsBar Oats Co...|      180g|  25891503|2026-03-03 09:07:...|products.csv|
|e92c739a8d78cd6cb...|      Nutrition Bars|      Energy Bars|SportsBar Energy ...|       40g|  25891102|2026-03-03 09:07:...|products.csv|
|d9ebd1ca64d23951a...|     Breakfast Foods|Granola & Cereals|SportsBar Granola...|      300g|  25891302|2026-03-03 09:07:...|products.csv|
|77b6f538a9d0e0cf8...|    Dairy & Recovery|   Recovery Dairy|SportsBar Greek Y...|       80g|  25891403|2026-03-03 09:07:...|products.csv|
|3cab59f0592428527...|     

In [0]:
df_silver.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .option("mergeSchema", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.silver.{data_source}")

In [0]:
df_gold = df_silver.select("product_code", "product_id", "division", "category", "product", "variant")
df_gold.show(5)

+--------------------+----------+----------------+-----------------+--------------------+-------+
|        product_code|product_id|        division|         category|             product|variant|
+--------------------+----------+----------------+-----------------+--------------------+-------+
|062f5574bbdf4386b...|  25891503|  Healthy Snacks|   Healthy Snacks|SportsBar Oats Co...|   180g|
|e92c739a8d78cd6cb...|  25891102|  Nutrition Bars|      Energy Bars|SportsBar Energy ...|    40g|
|d9ebd1ca64d23951a...|  25891302| Breakfast Foods|Granola & Cereals|SportsBar Granola...|   300g|
|77b6f538a9d0e0cf8...|  25891403|Dairy & Recovery|   Recovery Dairy|SportsBar Greek Y...|    80g|
|3cab59f0592428527...|  25891301| Breakfast Foods|Granola & Cereals|SportsBar Granola...|   400g|
+--------------------+----------+----------------+-----------------+--------------------+-------+
only showing top 5 rows


In [0]:
df_gold.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{catalog}.gold.sb_dim_{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark, "consumer_goods.gold.dim_products")
df_child_products = spark.sql(f"SELECT product_code, division, category, product, variant FROM consumer_goods.gold.sb_dim_products;")
df_child_products.show(5)

+--------------------+----------------+-----------------+--------------------+-------+
|        product_code|        division|         category|             product|variant|
+--------------------+----------------+-----------------+--------------------+-------+
|062f5574bbdf4386b...|  Healthy Snacks|   Healthy Snacks|SportsBar Oats Co...|   180g|
|e92c739a8d78cd6cb...|  Nutrition Bars|      Energy Bars|SportsBar Energy ...|    40g|
|d9ebd1ca64d23951a...| Breakfast Foods|Granola & Cereals|SportsBar Granola...|   300g|
|77b6f538a9d0e0cf8...|Dairy & Recovery|   Recovery Dairy|SportsBar Greek Y...|    80g|
|3cab59f0592428527...| Breakfast Foods|Granola & Cereals|SportsBar Granola...|   400g|
+--------------------+----------------+-----------------+--------------------+-------+
only showing top 5 rows


Merge

In [0]:
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
DESCRIBE HISTORY consumer_goods.gold.dim_customers

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-03-03T07:21:55.000Z,71320613334569,yvishnu224@gmail.com,MERGE,"Map(predicate -> [""(customer_code#15079 = customer_code#15057)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3430562047940079),61c4b1e8-57c9-4b59-81c6-7a39d62086b8,0303-070937-cpcqpwvy-v2n,1,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 2204, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 2994, materializeSourceTimeMs -> 4, numTargetRowsInserted -> 35, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1692, numTargetRowsUpdated -> 0, numOutputRows -> 35, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 35, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1141)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
1,2026-03-02T11:41:53.000Z,71320613334569,yvishnu224@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(903424024329876),e22cf9de-d132-4e16-bae5-1137f657d0ef,0302-113816-lp5wn68f-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1967, numDeletionVectorsRemoved -> 0, numOutputRows -> 18, numOutputBytes -> 1967)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
0,2026-02-28T06:18:09.000Z,71320613334569,yvishnu224@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(903424024329876),bf7c37c6-7670-4042-82a4-43f3bb8ea865,0228-052951-nxc2v7k3-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 18, numOutputBytes -> 1967)",null,Databricks-Runtime/18.0.x-aarch64-photon-scala2.13
